## Steps for pretraining an LLM from Scratch


1.	Define scope – use-case, languages, context length, quality bar.

2.	Collect & clean data – trillions of tokens ideally; dedup, filter toxicity, normalize text.

3.	Tokenizer – train BPE/Unigram (e.g., SentencePiece) on your corpus.

4.	Model architecture – Transformer (layers, heads, hidden size, RoPE, RMSNorm).

5.	Pretraining – self-supervised next-token prediction on large GPU/TPU clusters.

6.	Alignment – instruction tuning + preference optimization (SFT → DPO/RLHF).

7.	Evaluation – perplexity, benchmarks (MMLU, HELM), human eval.

8.	Optimization – pruning, quantization, distillation for inference.

9.	Serving – scalable inference, caching, safety filters, monitoring.


# Stages of LLM Training

## Overview

```
Raw Text Data
     ↓
1. Pre-training          → base model (knows language, facts)
     ↓
2. Supervised Fine-Tuning (SFT)  → instruction following
     ↓
3. Reward Modeling       → learns human preferences
     ↓
4. RLHF / PPO / DPO      → aligned, helpful, harmless model
     ↓
5. Evaluation & Red-teaming → safety & quality validation
```

---

## Stage 1 — Pre-Training

The foundation. The model learns language, facts, reasoning, and world knowledge by predicting the next token on massive text corpora — billions to trillions of tokens from the internet, books, code, papers.

**Objective:** Next token prediction (self-supervised, no labels needed)

```python
# Conceptually what happens during pre-training
input  = "The capital of France is"
target = "Paris"

# Model learns to predict every next token
# "The" → "capital"
# "The capital" → "of"
# "The capital of" → "France"
# "The capital of France" → "is"
# "The capital of France is" → "Paris"

loss = CrossEntropyLoss(predicted_token, actual_next_token)
```

**Data:** Common Crawl, Wikipedia, Books, GitHub, arXiv, StackOverflow — typically 1-15 trillion tokens

**Scale:**
- GPT-3: 300B tokens, 175B parameters
- LLaMA 3: 15 trillion tokens, up to 405B parameters
- Compute: thousands of GPUs for months

**Output:** A base model that completes text but doesn't follow instructions — ask it a question and it might just continue the sentence rather than answer.

---

## Stage 2 — Supervised Fine-Tuning (SFT)

Teaches the model to follow instructions and respond helpfully. Human annotators write high-quality examples of instruction → ideal response pairs. The model is fine-tuned on these.

**Objective:** Learn the format and style of helpful responses

```python
# Training examples look like this
examples = [
    {
        "instruction": "Explain photosynthesis simply",
        "response": "Photosynthesis is how plants make food using sunlight..."
    },
    {
        "instruction": "Write a Python function to reverse a string",
        "response": "def reverse_string(s):\n    return s[::-1]"
    },
    {
        "instruction": "Summarize this article: [article]",
        "response": "The article discusses..."
    }
]

# Fine-tune using PEFT/LoRA
from trl import SFTTrainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset,
    peft_config=lora_config,
    dataset_text_field="text"
)
trainer.train()
```

**Data size:** 10k to 1M high-quality instruction pairs — quality matters far more than quantity

**Output:** A model that follows instructions, answers questions, and has a helpful tone — but may still produce harmful, biased, or incorrect outputs

---

## Stage 3 — Reward Modeling

Trains a separate model to score responses based on human preferences. This reward model is later used to guide the main model's behavior.

**Process:**
1. Sample multiple responses from the SFT model for the same prompt
2. Human annotators rank the responses (which is best, worst)
3. Train a reward model to predict human preference scores

```python
# Human annotators rank these responses for "Explain black holes"
responses = {
    "A": "Black holes are regions of spacetime with gravity so strong...",  # ranked 1st
    "B": "A black hole is a hole that is black in space...",               # ranked 3rd
    "C": "Black holes form when massive stars collapse...",                 # ranked 2nd
}

# Reward model learns:
# score(A) > score(C) > score(B)

# Training uses pairwise ranking loss
loss = -log(sigmoid(reward(chosen) - reward(rejected)))
```

**Architecture:** Usually same as the SFT model with a linear head added that outputs a scalar score instead of token probabilities

**Output:** A reward model RM(prompt, response) → score that approximates human preference

---

## Stage 4 — Reinforcement Learning from Human Feedback (RLHF)

Uses the reward model to further train the SFT model via reinforcement learning. The model generates responses, the reward model scores them, and the model learns to generate higher-scoring responses.

```python
# PPO (Proximal Policy Optimization) training loop
for batch in training_data:
    # 1. Generate response with current policy
    response = policy_model.generate(prompt)
    
    # 2. Score with reward model
    reward = reward_model(prompt, response)
    
    # 3. KL penalty — don't drift too far from SFT model
    kl_penalty = kl_divergence(policy_model, sft_reference_model)
    
    # 4. Final reward
    final_reward = reward - beta * kl_penalty  # beta ~0.1-0.5
    
    # 5. Update policy to maximize reward
    ppo_update(policy_model, final_reward)
```

**The KL penalty is critical** — without it the model collapses into reward hacking, generating nonsense that tricks the reward model into giving high scores.

---

## Stage 4 (Alternative) — Direct Preference Optimization (DPO)

A simpler, more stable alternative to RLHF that skips the reward model entirely. Directly optimizes on human preference pairs without RL.

```python
from trl import DPOTrainer, DPOConfig

# Dataset of preference pairs
preference_data = [
    {
        "prompt":   "How do I stay focused?",
        "chosen":   "Try time-blocking, remove distractions...",  # preferred
        "rejected": "Just focus harder and stop being lazy."      # not preferred
    }
]

config = DPOConfig(
    beta=0.1,              # controls deviation from reference model
    learning_rate=5e-7,
    num_train_epochs=3
)

trainer = DPOTrainer(
    model=sft_model,
    ref_model=sft_reference,   # frozen reference
    args=config,
    train_dataset=preference_dataset
)

trainer.train()
```

**Why DPO is popular now:** Simpler to implement, more stable than PPO, no separate reward model needed, comparable or better results. Most modern open-source models (LLaMA 3, Mistral) use DPO or variants.

---

## Stage 4 (Variants) — Beyond RLHF and DPO

Several newer alignment approaches have emerged:

**ORPO (Odds Ratio Preference Optimization)** — combines SFT and preference learning in a single stage, no reference model needed, even simpler than DPO.

**KTO (Kahneman-Tversky Optimization)** — uses binary feedback (good/bad) instead of paired preferences, easier to collect data for.

**Constitutional AI (Anthropic)** — model critiques and revises its own outputs based on a set of principles, reducing reliance on human labelers.

**RLAIF (RL from AI Feedback)** — uses another LLM instead of humans to generate preference labels, enabling massive scale.

---

## Stage 5 — Evaluation & Red-Teaming

Before release, models are evaluated extensively for capability and safety.

```python
# Automated evaluation benchmarks
benchmarks = {
    "reasoning":    "MMLU, HellaSwag, ARC",
    "coding":       "HumanEval, MBPP",
    "math":         "GSM8K, MATH",
    "instruction":  "MT-Bench, AlpacaEval",
    "safety":       "TruthfulQA, BBQ"
}

# Red-teaming — humans try to break the model
red_team_categories = [
    "jailbreaks",           # bypassing safety guidelines
    "prompt injection",     # hijacking model behavior
    "harmful content",      # violence, hate speech
    "misinformation",       # factual errors
    "privacy violations"    # extracting training data
]
```

---

## Optional Stage — Continued Pre-Training

Sometimes done between pre-training and SFT to specialize the base model on domain-specific data before instruction tuning.

```python
# Example: Medical LLM
# Continue pre-training on medical literature
domain_data = [
    "pubmed_articles",
    "clinical_notes",
    "medical_textbooks",
    "drug_databases"
]
# Then SFT on medical Q&A pairs
# Then RLHF with medical expert feedback
```

Used for models like Med-PaLM, CodeLlama, and finance-specific LLMs.

---

## Full Pipeline Summary

| Stage | Goal | Data | Key Challenge |
|---|---|---|---|
| Pre-training | Learn language & knowledge | Trillions of tokens | Compute cost |
| SFT | Follow instructions | 10k-1M labeled pairs | Data quality |
| Reward Modeling | Learn human preferences | 100k ranked pairs | Annotation consistency |
| RLHF/DPO | Align with human values | Preference pairs | Reward hacking / stability |
| Evaluation | Validate safety & quality | Benchmarks + red team | Coverage of failure modes |

---

## Compute & Cost Reality

| Stage | Relative Cost | Duration (7B model) |
|---|---|---|
| Pre-training | 99% of total cost | Weeks to months |
| SFT | ~0.5% | Hours to days |
| Reward Modeling | ~0.3% | Hours to days |
| RLHF/DPO | ~0.2% | Hours to days |

Pre-training dominates entirely — it's why only well-funded labs train from scratch. Most organizations start from an open-source base model (LLaMA, Mistral) and do SFT + DPO, which is affordable on a single GPU cluster.